In [ ]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({
    "font.family" : "serif",
    "font.size" : 15,
    "mathtext.fontset" : "stix",
    "font.serif" : ['STIXGeneral']
})

In [ ]:

folderVec = [
    "/Users/maggie/repo/LBM-Program/amrlbm/bin/acoustic_Re150/cylinderRe150_cml_m02_25dx_test_2_75D/probes/probe2",
    "/Users/maggie/repo/LBM-Program/amrlbm/bin/acoustic_Re150/cylinderRe150_cml_m02_40dx_test_3_75D/probes/probe2",
    "/Users/maggie/repo/LBM-Program/amrlbm/bin/acoustic_Re150/cylinderRe150_bgk_m02_40dx_test_75D/probes/probe2"
]

rho0 = 1.0
cs2 = 1.0 / 3.0  # lattice speed of sound squared

U0Vec = [
    0.115470054,
    0.115470054,
    0.115470054,
]

startStep = 40000

avgStepRg = [
    [80000, 100000],
    [80000, 100000],
    [80000, 100000],
]

colorVec = [
    "#05A361",
    "#D33737",
    "#001AFF",
    '#FF5733',
    "#119100",
    "#B700FF",
]

markerVec = [
    'o',
    '^',
    's',
    "P",
    '*',
]

labelVec = [
    # "Present LBM",
    "Present LBM Cumulant D/dx=25",
    "Present LBM Cumulant D/dx=40",
    "Present LBM BGK D/dx=40",
]

In [ ]:
def read_col_probe(filePath, colIdx, skipHeader=2):
    with open(filePath, 'r') as f:
        vec = np.genfromtxt(f, skip_header=skipHeader, usecols=colIdx)
        vec = vec[~np.isnan(vec)]
    f.close()
    return vec

def get_probe_coords(filePath):
    with open(filePath, 'r') as f:
        header = f.readline()
        coordsStr = header.split('(')[1].split(')')[0].split(',')
        x = float(coordsStr[0])
        y = float(coordsStr[1])
        z = float(coordsStr[2])
    return x,y,z

def read_csv_col(filePath, colIdx, skipHeader=1):
    with open(filePath, 'r') as f:
        vec = np.genfromtxt(f, delimiter=',', skip_header=skipHeader, usecols=colIdx)
    f.close()
    return vec


In [ ]:

thetaVecVec = []
pCoefVecVec = []
for case in range(0, len(folderVec)):
    files = sorted(glob.glob(os.path.join(folderVec[case], "probe*.txt")))
    nProbes = len(files)

    stepCol = read_col_probe(files[0], 0, 2)
    idxStart = int(np.abs(stepCol - avgStepRg[case][0]).argmin())
    idxEnd = int(np.abs(stepCol - avgStepRg[case][1]).argmin())
    thetaVec = []
    pCoefVec = []
    dynPressure = 0.5 * rho0 * U0Vec[case]**2
    for iProbe in range(0, len(files)):
        x, y, z = get_probe_coords(files[iProbe])
        theta = np.degrees(np.arctan2(y, x)) + 180
        thetaVec.append(theta)
        rhoCol = read_col_probe(files[iProbe], 2, 2)
        pressure = (rhoCol - rho0) * cs2
        avgPressure = np.mean(pressure[idxStart:idxEnd])
        pCoef = avgPressure / dynPressure
        pCoefVec.append(pCoef)
    thetaVecVec.append(thetaVec)
    pCoefVecVec.append(pCoefVec)
    

In [ ]:
refCpFile1 = "../ref/cp_inoue_Ma02.csv"
refTheta1 = read_csv_col(refCpFile1, 0, 1)
refCp1 = read_csv_col(refCpFile1, 1, 1)

refCpFile2 = "../ref/cp_kusano_Ma02.csv"
refTheta2 = read_csv_col(refCpFile2, 0, 1)
refCp2 = read_csv_col(refCpFile2, 1, 1)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(7, 6), facecolor='w', edgecolor='w')

ax.plot(refTheta1, refCp1, lw=1.5, c='k', label='Inoue & Hatakeyama, DNS')
ax.plot(refTheta2, refCp2, lw=1.5, c='k', linestyle='-.', label='Kusano et al., LBM BGK athermal filtered')
for case in range(0, len(folderVec)):
    ax.plot(thetaVecVec[case][:90], pCoefVecVec[case][:90], c=colorVec[case], lw=0, marker=markerVec[case], fillstyle='none', ms=6, mew=1.5, markevery=2, label=labelVec[case])

ax.set_xlabel(r"$\theta$")
ax.set_ylabel(r"$C_p$")
ax.set_xlim(0, 180)
ax.set_ylim(-2, 1.5)
ax.tick_params(width=2.0, axis='both', direction='in')
for spine in ax.spines.values():
    spine.set_linewidth(2.0)

ax.legend(loc='upper right', frameon=False, fontsize=15)